# 05 — Validation analysis and model comparison

Uses **validation participants only** (P007, P008). Everything here informs the final choice; nothing here looks
at the test set. Run notebook 04 first (at least one Stage A run; ideally a Stage B run too).

Figures go to `experiments/validation_analysis_<NNN>/figures/` as PNG and SVG.

**Statistical caution**: two validation participants can rank models, but cannot estimate how accurate the model
will be for a new person. Confidence intervals below are *conditional on these two people*.

In [ ]:
# --- Workspace setup -----------------------------------------------------------
# WHAT: makes the `src` package importable and loads training_config.yaml.
# WHY : every notebook uses the same code and the same configuration.
# PATHS: resolved from this notebook's folder, never from the terminal's cwd.
import sys
from pathlib import Path

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
if not (WORKSPACE / "src" / "config.py").is_file():
    raise RuntimeError(f"Open this notebook from training_workspace/notebooks (cwd = {Path.cwd()})")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.config import load_config
cfg = load_config(WORKSPACE / "training_config.yaml")
cfg.paths.ensure_output_dirs()
W, H = cfg.screen_px
print("Workspace     :", cfg.paths.workspace)
print("Dataset root  :", cfg.paths.dataset_root, "(read-only)")
print("Screen        :", W, "x", H, "px")

In [ ]:
import json, numpy as np, pandas as pd, torch, yaml
from IPython.display import Image, display
from src.utils import new_run_dir, write_json, select_device
from src.visualization import (plot_comparison, plot_prediction_map, plot_error_heatmap, plot_vector_field,
                               plot_predicted_coverage, interactive_prediction_map, replay_animation,
                               inspect_sample, plot_error_analysis)
from src.dataset import load_split
va_df = load_split(cfg, "validation")
out = new_run_dir(cfg.paths.experiments, "validation_analysis")
runs = []
for p in sorted(cfg.paths.experiments.glob("*/run_info.json")):
    i = json.loads(p.read_text())
    if "best_val_metric" not in i or p.parent.name.startswith("datacurve_"):
        continue                     # data-size experiments are analysed in section 2 only
    c = yaml.safe_load((p.parent / "config.yaml").read_text())
    runs.append({"run": p.parent.name, "stage": i["stage"], "val_mean_px_macro": i["best_val_metric"],
                 "best_epoch": i["best_epoch"], "unfrozen_blocks": i.get("unfrozen_blocks", 0),
                 "augmentation": c["augmentation"]["enabled"], "head": c["model"]["head"]["type"],
                 "trainable_parameters": i.get("trainable_parameters")})
runs_df = pd.DataFrame(runs).sort_values("val_mean_px_macro")
runs_df

## 1. Checkpoint / stage comparison (frozen vs fine-tuned, unfrozen blocks, augmentation, head type)

In [ ]:
if len(runs_df):
    figs = [plot_comparison(runs_df, "run", "val_mean_px_macro", out / "figures" / "checkpoint_comparison",
                            "Best validation error per run"),
            plot_comparison(runs_df.groupby("unfrozen_blocks", as_index=False)["val_mean_px_macro"].min(),
                            "unfrozen_blocks", "val_mean_px_macro", out / "figures" / "error_vs_unfrozen_blocks",
                            "Validation error vs number of unfrozen blocks (0 = frozen)", kind="line"),
            plot_comparison(runs_df.groupby("augmentation", as_index=False)["val_mean_px_macro"].min(),
                            "augmentation", "val_mean_px_macro", out / "figures" / "augmentation_comparison",
                            "Augmentation on vs off (best run of each)")]
    for f in figs: display(Image(filename=str(f)))

## 2. Performance vs amount of training data (learning-curve experiment) — opt-in, fast (head only)
Re-trains the Stage A head on 10 / 25 / 50 / 100 % of each training participant's **trials** (frames of a trial
stay together) using cached features. If error is still dropping steeply at 100 %, more data per person would help;
if it is flat, more *people* (or better features) are needed instead.

In [ ]:
RUN_DATA_CURVE = False
if RUN_DATA_CURVE:
    from src.models import load_unigaze, CoordinateHead
    from src.pipeline import extract_split_features, head_inputs, split_source
    from src.trainer import train_head_on_features
    dev = select_device(cfg["hardware"]["device"], cfg["hardware"]["allow_cpu_training"], cfg["hardware"]["precision"])
    unigaze, ck = load_unigaze(cfg)
    tr_df, _, _ = split_source(cfg, "train"); vdf, _, _ = split_source(cfg, "validation")
    ftr = extract_split_features(cfg, unigaze, ck["sha256"], "train", dev.device, dev.precision)
    fva = extract_split_features(cfg, unigaze, ck["sha256"], "validation", dev.device, dev.precision)
    use = cfg["model"]["head"]["use_gaze_angles"]
    Xtr, Xva = head_inputs(ftr, use), head_inputs(fva, use)
    vu = np.asarray(fva["usable"]); rng = np.random.default_rng(0)
    curve = []
    for frac in (0.1, 0.25, 0.5, 1.0):
        keep_trials = set()
        for pid, g in tr_df.groupby("participant_id"):
            t = g["trial_id"].unique(); keep_trials |= set(rng.choice(t, max(1, int(frac * len(t))), replace=False))
        m = tr_df["trial_id"].isin(keep_trials).to_numpy() & np.asarray(ftr["usable"])
        head = CoordinateHead(Xtr.shape[1], cfg["model"]["head"]["type"], cfg["model"]["head"]["hidden_dim"])
        r = train_head_on_features(cfg, head, Xtr[m], tr_df[m].reset_index(drop=True), Xva[vu],
                                   vdf[vu].reset_index(drop=True), new_run_dir(cfg.paths.experiments, f"datacurve_{int(frac*100)}pct"),
                                   dev.device, "stage_a", {"data_fraction": frac})
        curve.append({"train_trials": len(keep_trials), "fraction": frac, "val_mean_px_macro": r["best_val_mean_euclid_px_macro"]})
    curve = pd.DataFrame(curve); curve.to_csv(out / "learning_curve_data_size.csv", index=False)
    display(Image(filename=str(plot_comparison(curve, "train_trials", "val_mean_px_macro",
            out / "figures" / "error_vs_training_trials", "Validation error vs training trials", kind="line"))))

## 3. Detailed analysis of the selected run
`SELECTED_RUN` defaults to the best run by validation error. Its all-frame validation predictions were written by
notebook 04 §27 (`val_predictions_all_frames.parquet`).

In [ ]:
have_preds = [r for r in runs_df["run"] if (cfg.paths.experiments / r / "val_predictions_all_frames.parquet").is_file()] if len(runs_df) else []
SELECTED_RUN = have_preds[0] if have_preds else None      # best run that has all-frame predictions
pred = pd.read_parquet(cfg.paths.experiments / SELECTED_RUN / "val_predictions_all_frames.parquet") if SELECTED_RUN else None
print("selected:", SELECTED_RUN, "| frames:", None if pred is None else len(pred))
if SELECTED_RUN is None:
    print("No run has all-frame validation predictions yet - run notebook 04 sections 27-28 for your chosen run.")

## 4. Per-participant comparison across runs

In [ ]:
rows = []
for r in runs_df["run"]:
    p = cfg.paths.experiments / r / "val_predictions_all_frames.parquet"
    if p.is_file():
        d = pd.read_parquet(p)
        for pid, g in d.groupby("participant_id"):
            rows.append({"run": r, "participant_id": pid, "mean_euclid_px": g["euclid_px"].mean()})
if rows:
    pp = pd.DataFrame(rows)
    display(Image(filename=str(plot_comparison(pp, "run", "mean_euclid_px", out / "figures" / "per_participant_comparison",
                                               "Per-participant validation error", hue="participant_id"))))
    pp.pivot(index="run", columns="participant_id", values="mean_euclid_px").round(1)

## 5. Error analysis figures (histogram, CDF, participant/session box plots, residuals, region, quality)

In [ ]:
if pred is not None:
    for f in plot_error_analysis(pred, W, H, out / "figures", "validation", manifest=va_df).values():
        display(Image(filename=str(f)))

## 6. Prediction maps on the participant monitor (2560×1600, origin top-left, y downward)
Black ✕ = instructed target; coloured dot = prediction (trial mean); line = error.

In [ ]:
if pred is not None:
    figs = [plot_prediction_map(pred, W, H, out / "figures" / "map_all", "Validation: all participants"),
            plot_error_heatmap(pred, W, H, out / "figures" / "map_error_heatmap"),
            plot_vector_field(pred, W, H, out / "figures" / "map_vector_field"),
            plot_predicted_coverage(pred, W, H, out / "figures" / "map_coverage")]
    for pid, g in pred.groupby("participant_id"):
        figs.append(plot_prediction_map(g, W, H, out / "figures" / f"map_{pid}", f"Validation {pid}", color_by="session_id"))
    for (pid, sid), g in pred.groupby(["participant_id", "session_id"]):
        plot_prediction_map(g, W, H, out / "figures" / f"map_{pid}_{sid}", f"Validation {pid} {sid}", color_by="session_id")
    for f in figs: display(Image(filename=str(f)))
    html = interactive_prediction_map(pred, W, H, out / "interactive_prediction_map_validation.html", f"Validation - {SELECTED_RUN}")
    print("Interactive map (open in a browser):", html)

## 7. Sample-by-sample inspection and offline replay

In [ ]:
if pred is not None:
    worst = pred.sort_values("euclid_px", ascending=False).head(3)["sample_id"].tolist()
    for sid in worst + [pred.sample(1, random_state=0)["sample_id"].item()]:
        inspect_sample(pred, va_df, cfg.paths.dataset_root, sid, W, H)
    first = pred[["participant_id", "session_id"]].iloc[0]
    print("replay:", replay_animation(pred, W, H, out / "replay.html", first.participant_id, first.session_id))

## 8. Calibration vs no calibration (validation)
Calibrate on S01 (k points), score on S02–S05. Paired comparison on the same frames. Reported separately from the
uncalibrated benchmark.

In [ ]:
from src.calibration import evaluate_calibration
if pred is not None:
    cal = evaluate_calibration(pred, cfg, cfg["calibration"]["point_counts"])
    cal.to_csv(out / "calibration_validation.csv", index=False)
    display(Image(filename=str(plot_comparison(cal, "k_points", "mean_euclid_px", out / "figures" / "calibration_vs_none",
                "Calibration points vs error (0 = uncalibrated)", hue="participant_id", kind="line"))))
    cal

## 9. Temporal smoothing — jitter vs accuracy (within trials only)
Filters run inside each trial in capture-time order; they never cross trials, sessions or participants.
This dataset has no frames during target transitions, so response delay/overshoot cannot be measured here
(see `src/smoothing.py → TRANSITION_PROTOCOL`).

In [ ]:
from src.smoothing import FILTERS, smooth_predictions, fixation_jitter
from src.metrics import regression_metrics
if pred is not None:
    rows = []
    for name, make in FILTERS.items():
        s = smooth_predictions(pred, make)
        m = regression_metrics(s[["pred_x_norm", "pred_y_norm"]].to_numpy(), s[["target_x_norm", "target_y_norm"]].to_numpy(), W, H)
        rows.append({"filter": name, **fixation_jitter(s, W, H), "mean_euclid_px": m["mean_euclid_px"]})
    smooth_tbl = pd.DataFrame(rows); smooth_tbl.to_csv(out / "smoothing_validation.csv", index=False); smooth_tbl.round(1)

## 10. Freeze the decision (before opening the test set)
Write down which run is final and why. Notebook 06 refuses to run without `reports/model_selection_decision.json`
(written by notebook 04 §30 when you export).

In [ ]:
print("Selected (lowest validation error):", SELECTED_RUN)
print("Export it in notebook 04, section 30, then open notebook 06.")